# Google AlloyDB for PostgreSQL - `AlloyDBEmbedding`

> [AlloyDB](https://cloud.google.com/alloydb) is a fully managed relational database service that offers high performance, seamless integration, and impressive scalability. AlloyDB is 100% compatible with PostgreSQL. Extend your database application to build AI-powered experiences leveraging AlloyDB's LlamaIndex integrations.

This notebook goes over how to generate embeddings inside AlloyDB with the `AlloyDBEmbedding` class, and how to use it with a LlamaIndex `VectorStoreIndex`.

Learn more about the package on [GitHub](https://github.com/googleapis/llama-index-alloydb-pg-python/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googleapis/llama-index-alloydb-pg-python/blob/main/samples/alloydb_embeddings.ipynb)

## Before you begin

To run this notebook, you will need to do the following:

 * [Create a Google Cloud Project](https://developers.google.com/workspace/guides/create-project)
 * [Enable the AlloyDB API](https://console.cloud.google.com/flows/enableapi?apiid=alloydb.googleapis.com)
 * [Create a AlloyDB cluster and instance.](https://cloud.google.com/alloydb/docs/cluster-create)
 * [Create a AlloyDB database.](https://cloud.google.com/alloydb/docs/quickstart/create-and-connect)
 * [Add a User to the database.](https://cloud.google.com/alloydb/docs/database-users/about)
 * [Set up embedding generation](https://cloud.google.com/alloydb/docs/ai/work-with-embeddings) with the `google_ml_integration` extension, so the instance can call your embedding model (for example `text-embedding-005`).

### 🦙 Library Installation
Install the integration library, `llama-index-alloydb-pg`.

In [ ]:
%pip install --upgrade --quiet llama-index-alloydb-pg

### 🔐 Authentication
Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

* If you are using Colab to run this notebook, use the cell below and continue.
* If you are using Vertex AI Workbench, check out the setup instructions [here](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env).

In [ ]:
from google.colab import auth

auth.authenticate_user()

### Set AlloyDB database values
Find your database values, in the [AlloyDB Instances page](https://console.cloud.google.com/alloydb/clusters).

In [ ]:
# @title Set Your Values Here { display-mode: "form" }
PROJECT_ID = "my-project-id"  # @param {type: "string"}
REGION = "us-central1"  # @param {type: "string"}
CLUSTER = "my-cluster"  # @param {type: "string"}
INSTANCE = "my-primary"  # @param {type: "string"}
DATABASE = "my-database"  # @param {type: "string"}
TABLE_NAME = "vector_store"  # @param {type: "string"}
USER = "postgres"  # @param {type: "string"}
PASSWORD = "my-password"  # @param {type: "string"}

### Create an `AlloyDBEngine`

The `AlloyDBEngine` configures a connection pool to your AlloyDB database.

In [ ]:
from llama_index_alloydb_pg import AlloyDBEngine

engine = await AlloyDBEngine.afrom_instance(
    project_id=PROJECT_ID,
    region=REGION,
    cluster=CLUSTER,
    instance=INSTANCE,
    database=DATABASE,
    user=USER,
    password=PASSWORD,
)

## Generate embeddings

`AlloyDBEmbedding` calls the AlloyDB `embedding()` function, so the embedding model runs from the database. `model_id` is the id of a model known to AlloyDB model endpoint management, for example `text-embedding-005`.

In [ ]:
from llama_index_alloydb_pg import AlloyDBEmbedding

embed_model = AlloyDBEmbedding(engine=engine, model_id="text-embedding-005")

query_embedding = await embed_model.aget_query_embedding("What is AlloyDB?")
print(len(query_embedding))

text_embeddings = await embed_model.aget_text_embedding_batch(
    ["AlloyDB is a PostgreSQL-compatible database.", "Paris is the capital of France."]
)
print(len(text_embeddings))

## Use it with a `VectorStoreIndex`

Use `AlloyDBEmbedding` as the `embed_model` of an index backed by `AlloyDBVectorStore`. `text-embedding-005` returns 768 dimensions, so the vector store table uses `vector_size=768`.

In [ ]:
from llama_index.core import Document, StorageContext, VectorStoreIndex

from llama_index_alloydb_pg import AlloyDBVectorStore

await engine.ainit_vector_store_table(table_name=TABLE_NAME, vector_size=768)
vector_store = await AlloyDBVectorStore.create(engine=engine, table_name=TABLE_NAME)

documents = [
    Document(text="AlloyDB is a PostgreSQL-compatible database."),
    Document(text="Paris is the capital of France."),
]
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=StorageContext.from_defaults(vector_store=vector_store),
    embed_model=embed_model,
)

retriever = index.as_retriever(similarity_top_k=1)
nodes = await retriever.aretrieve("Which city is the capital of France?")
print(nodes[0].node.get_content())